[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Tensors &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


**1.** Make the whole numbers 1 to 10 as a PyTorch tensor with `torch.arange` and as a TensorFlow tensor with `tf.range`, and print each one's dtype.

In [2]:
counts = torch.arange(1, 11)
counts_tf = tf.range(1, 11)
print(counts, counts.dtype)
print(counts_tf.numpy(), counts_tf.dtype)

tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]) torch.int64
[ 1  2  3  4  5  6  7  8  9 10] <dtype: 'int32'>


`torch.arange` makes 64-bit integers and `tf.range` 32-bit ones, the defaults for whole numbers. Both stop before 11.

**2.** Make a TensorFlow tensor of the first five January means in `jan_mean` as 64-bit floats, cast it to 32-bit floats, and print both dtypes and the cast values.

In [3]:
wide = tf.constant(jan_mean[:5], dtype=tf.float64)
narrow = tf.cast(wide, tf.float32)
print(wide.dtype, narrow.dtype)
print(narrow.numpy())

<dtype: 'float64'> <dtype: 'float32'>
[-9.6 -3.7 -4.8 -9.4 -8.2]


`dtype=tf.float64` makes the 64-bit tensor, and `tf.cast` makes the 32-bit one from it.

**3.** Draw four numbers from a normal distribution in PyTorch from the seed 42, twice, and check with `torch.equal` that the two draws match. Then do the same in TensorFlow with `tf.random.Generator.from_seed(42)` and `tf.reduce_all(a == b)`.

In [4]:
torch.manual_seed(42)
first = torch.randn(4)
torch.manual_seed(42)
second = torch.randn(4)
print(torch.equal(first, second))

a = tf.random.Generator.from_seed(42).normal((4,))
b = tf.random.Generator.from_seed(42).normal((4,))
print(bool(tf.reduce_all(a == b)))

True
True


Both pairs of draws match, because each draw started from the same seed.

**4.** Make a `tf.Variable` holding the first five January means, add 1.0 to the third of them with `assign`, and print the variable.

In [5]:
five = tf.Variable(jan_mean[:5])
five[2].assign(five[2] + 1.0)
print(five.numpy().round(1))
print(jan_mean[:5])

[-9.6 -3.7 -3.8 -9.4 -8.2]
[-9.6 -3.7 -4.8 -9.4 -8.2]


Only the variable changed. `jan_mean` keeps −4.8 in its third place, because `tf.Variable` copied the array.

**5.** Make `sample = scaled[:3].copy()`, then one tensor from it with `torch.from_numpy` and one with `torch.tensor`. Set `sample[0, 0] = 0.0` and print the first value of each tensor.

In [6]:
sample = scaled[:3].copy()
linked = torch.from_numpy(sample)
separate = torch.tensor(sample)
print(round(float(sample[0, 0]), 3))
sample[0, 0] = 0.0
print(round(linked[0, 0].item(), 3), round(separate[0, 0].item(), 3))

1.754
0.0 1.754


Changing `sample` changed the tensor from `torch.from_numpy`, which shares its memory, and left the one from `torch.tensor` at 1.754.

---

&#8592; **Back to:** [Tensors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/02-tensors.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
